### Testing
### Unit Tests – Format Conversion Validation

### Objective
This notebook validates the correctness of CSV to JSON and CSV to XML
format conversions performed during the data ingestion phase.
The goal is to ensure that no data loss or schema inconsistencies
occur during format transformation.

### Scope
The following datasets are validated:
- `menu_items` (CSV → JSON)
- `stores` (CSV → XML)

#### Tests for menu_items (CSV → JSON)

In [0]:
# Row count check
df_menu_csv = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/coffee_raw_volume/menu_items/")
)

df_menu_json = (
    spark.read
    .json("/Volumes/workspace/default/coffee_raw_volume/menu_items_json/")
)

print("Menu CSV count:", df_menu_csv.count())
print("Menu JSON count:", df_menu_json.count())


In [0]:
# Column presence check
csv_columns = set(df_menu_csv.columns)
json_columns = set(df_menu_json.columns)

print("Missing in JSON:", csv_columns - json_columns)
print("Extra in JSON:", json_columns - csv_columns)


In [0]:
# Null checks
from pyspark.sql.functions import col, sum

df_menu_json.select(
    sum(col("item_id").isNull().cast("int")).alias("null_item_id"),
    sum(col("item_name").isNull().cast("int")).alias("null_item_name"),
    sum(col("price").isNull().cast("int")).alias("null_price")
).show()


In [0]:
# Duplicate key check
df_menu_json.groupBy("item_id").count().filter("count > 1").show()


#### Tests for stores (CSV → XML)

In [0]:
# row count match 
df_stores_csv = (
    spark.read
    .option("header", True)
    .csv("/Volumes/workspace/default/coffee_raw_volume/stores/")
)

df_stores_xml = (
    spark.read
    .format("xml")
    .option("rowTag", "store")
    .load("/Volumes/workspace/default/coffee_raw_volume/stores_xml/")
)

print("Stores CSV count:", df_stores_csv.count())
print("Stores XML count:", df_stores_xml.count())


In [0]:
# column checks
csv_columns = set(df_stores_csv.columns)
xml_columns = set(df_stores_xml.columns)

print("Missing in XML:", csv_columns - xml_columns)
print("Extra in XML:", xml_columns - csv_columns)


In [0]:
#Null checks
df_stores_xml.select(
    sum(col("store_id").isNull().cast("int")).alias("null_store_id"),
    sum(col("store_name").isNull().cast("int")).alias("null_store_name"),
    sum(col("city").isNull().cast("int")).alias("null_city")
).show()


In [0]:
# Duplicate id
df_stores_xml.groupBy("store_id").count().filter("count > 1").show()
